# Lab 6: LLM Architecture Anatomy

## What this notebook does

You finish `ModelConfig`, a dataclass that holds a transformer's design numbers (`vocab_size`, `d_model`, `n_layers`, `n_heads`, `d_ff`). You write methods that count how many parameters each part has: the embedding, attention, feed-forward, layer norms and output head. `count_parameters` adds them up into a total, with one entry for each part.

The reading pane above explains parameter counting, dataclasses as config objects, and model config dicts in full.

In [ ]:
from dataclasses import dataclass

## Exercise

### Goal
Big language models are mostly big piles of numbers called **parameters**. A parameter is one learnable number inside the model. "7 billion parameters" means 7 billion such numbers. Before you train or run a model, you can work out how many it has from its settings, using only multiplication. You finish a small `ModelConfig` class. It counts the parameters in each part of a simple transformer language model (the same kind of model behind chatbots). The class adds them up for you.

What you start with: `ModelConfig` already has `embedding_params` and `count_parameters` written. `count_parameters` calls the four methods you write, and multiplies the per-block ones by `n_layers`. You only fill in four small methods. Each takes only `self` and gives back an `int`.

Here are the settings. `ModelConfig` is a *dataclass*, so its fields are listed at the top and stored for you. You read them as `self.d_model` and so on.
- `vocab_size`: how many different tokens (word pieces) the model knows.
- `d_model`: the width of the model. It is how many numbers stand for one token inside it.
- `d_ff`: the width of the feed-forward layer's inner step (usually about 4 times `d_model`).
- `n_layers`: how many identical transformer **blocks** are stacked. Each block has its own attention, feed-forward and layer-norm parameters.
- `n_heads`: how many attention heads. It is stored in the config, but it does **not** change the count in this exercise.

A matrix of shape `a x b` is a table of numbers with `a` rows and `b` columns. It holds `a * b` parameters.

Running example used below: `tiny = ModelConfig(vocab_size=1000, d_model=64, n_layers=2, n_heads=4, d_ff=256)`. There is also a micro model, `ModelConfig(vocab_size=10, d_model=4, n_layers=1, n_heads=2, d_ff=8)`, for checks by hand.

### 1. `attention_params_per_layer(self)`
Code shape (replace each `...`):
```python
def attention_params_per_layer(self):
    return ...
```
**You give back:** an `int`, the attention parameters in **one** block.
**Idea:** *attention* is the part where tokens look at each other. It uses 4 square matrices (query, key, value and output). Each is `d_model` by `d_model`. So the answer is 4 times `d_model` times `d_model`. Ignore bias vectors.
**Check:** micro model: `4 * 4 * 4 = 64`. For `tiny`: `4 * 64 * 64 = 16384`.

### 2. `feed_forward_params_per_layer(self)`
Code shape:
```python
def feed_forward_params_per_layer(self):
    first = ...
    second = ...
    return ...
```
**You give back:** an `int`, the feed-forward parameters in **one** block.
**Idea:** the *feed-forward* part is two layers in a row. The first layer is a `d_model x d_ff` matrix. The second is `d_ff x d_model`. Add the two sizes. They are equal, so it is also twice one of them.
**Check:** micro model: `4*8 + 8*4 = 64`. For `tiny`: `64*256*2 = 32768`.

### 3. `layernorm_params_per_layer(self)`
Code shape:
```python
def layernorm_params_per_layer(self):
    one_norm = ...
    return ...
```
**You give back:** an `int`, the layer-norm parameters in **one** block.
**Idea:** a *layer norm* is a small tidying step. Each one has a scale and a shift for every dimension, so it has `2 * d_model` numbers. Each block has 2 layer norms.
**Check:** micro model: `2 * 2 * 4 = 16`. For `tiny`: `2 * 2 * 64 = 256`.

### 4. `output_head_params(self)`
Code shape:
```python
def output_head_params(self):
    return ...
```
**You give back:** an `int`, the size of the final `d_model x vocab_size` matrix. This matrix is the *output head*. It scores every token. It does not depend on `n_layers`. Treat it as its own matrix, not shared with the embedding table.
**Check:** micro model: `4 * 10 = 40`. For `tiny`: `64 * 1000 = 64000`.

### Putting it together (already coded)
For `tiny`, once your methods are right, `tiny.count_parameters()` gives back
`{'embedding': 64000, 'attention': 32768, 'feed_forward': 65536, 'layernorm': 512, 'output_head': 64000, 'total': 226816}`.
The per-block numbers are doubled because `n_layers=2`. The *embedding table* is a `vocab_size x d_model` table that turns a token id into its list of numbers. It is already written for you. For the micro model the total is `40 + 64 + 64 + 16 + 40 = 224`.

### What the asserts below test
The asserts build `tiny` and compare each of your four methods with the formula (`4*64*64`, `64*256*2`, `2*2*64`, `64*1000`). An *assert* is a line that stops the program with a message if its condition is false. A failed assert shows the expected value and the value you got. If you see `All checks passed.`, all four counts are right. The cell then prints the full parameter breakdown dict.

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int
    d_model: int
    n_layers: int
    n_heads: int
    d_ff: int

    def embedding_params(self) -> int:
        return self.vocab_size * self.d_model

    def attention_params_per_layer(self) -> int:
        """Count the parameters in the attention part of ONE block.

        You give back: an int. Example: d_model=64 -> 16384.
        """
        # TODO:
        # 1. Attention has 4 square matrices (query, key, value and output).
        # 2. Each one is d_model x d_model, so it holds d_model * d_model numbers.
        # 3. Return 4 times that. Ignore bias vectors.
        raise NotImplementedError("ModelConfig.attention_params_per_layer() is not implemented yet - replace the TODO with your code")

    def feed_forward_params_per_layer(self) -> int:
        """Count the parameters in the feed-forward part of ONE block.

        You give back: an int. Example: d_model=64, d_ff=256 -> 32768.
        """
        # TODO:
        # 1. The first layer has d_model x d_ff numbers.
        # 2. The second layer has d_ff x d_model numbers.
        # 3. Return the two sizes added together.
        raise NotImplementedError("ModelConfig.feed_forward_params_per_layer() is not implemented yet - replace the TODO with your code")

    def layernorm_params_per_layer(self) -> int:
        """Count the parameters in the layer norms of ONE block.

        You give back: an int. Example: d_model=64 -> 256.
        """
        # TODO:
        # 1. One layer norm has 2 * d_model numbers (a scale and a shift).
        # 2. A block has 2 layer norms.
        # 3. Return the total for the block.
        raise NotImplementedError("ModelConfig.layernorm_params_per_layer() is not implemented yet - replace the TODO with your code")

    def output_head_params(self) -> int:
        """Count the parameters in the final output head (counted once, not per block).

        You give back: an int. Example: d_model=64, vocab_size=1000 -> 64000.
        """
        # TODO:
        # 1. The head is a d_model x vocab_size matrix.
        # 2. It is its own matrix (not shared with the embedding table), so count all of it.
        # 3. Return d_model * vocab_size.
        raise NotImplementedError("ModelConfig.output_head_params() is not implemented yet - replace the TODO with your code")

    def count_parameters(self) -> dict[str, int]:
        per_layer = (
            self.attention_params_per_layer()
            + self.feed_forward_params_per_layer()
            + self.layernorm_params_per_layer()
        )
        breakdown = {
            "embedding": self.embedding_params(),
            "attention": self.attention_params_per_layer() * self.n_layers,
            "feed_forward": self.feed_forward_params_per_layer() * self.n_layers,
            "layernorm": self.layernorm_params_per_layer() * self.n_layers,
            "output_head": self.output_head_params(),
        }
        breakdown["total"] = sum(v for k, v in breakdown.items() if k != "total")
        return breakdown


tiny = ModelConfig(vocab_size=1000, d_model=64, n_layers=2, n_heads=4, d_ff=256)
assert tiny.attention_params_per_layer() == 4 * 64 * 64, 'tiny.attention_params_per_layer() should be ' + repr(4 * 64 * 64) + ", got " + repr(tiny.attention_params_per_layer())
assert tiny.feed_forward_params_per_layer() == 64 * 256 * 2, 'tiny.feed_forward_params_per_layer() should be ' + repr(64 * 256 * 2) + ", got " + repr(tiny.feed_forward_params_per_layer())
assert tiny.layernorm_params_per_layer() == 2 * 2 * 64, 'tiny.layernorm_params_per_layer() should be ' + repr(2 * 2 * 64) + ", got " + repr(tiny.layernorm_params_per_layer())
assert tiny.output_head_params() == 64 * 1000, 'tiny.output_head_params() should be ' + repr(64 * 1000) + ", got " + repr(tiny.output_head_params())

print("All checks passed.")
print(tiny.count_parameters())

## Solution

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int
    d_model: int
    n_layers: int
    n_heads: int
    d_ff: int

    def embedding_params(self) -> int:
        return self.vocab_size * self.d_model

    def attention_params_per_layer(self) -> int:
        return 4 * self.d_model * self.d_model

    def feed_forward_params_per_layer(self) -> int:
        return self.d_model * self.d_ff + self.d_ff * self.d_model

    def layernorm_params_per_layer(self) -> int:
        return 2 * 2 * self.d_model

    def output_head_params(self) -> int:
        return self.d_model * self.vocab_size

    def count_parameters(self) -> dict[str, int]:
        per_layer = (
            self.attention_params_per_layer()
            + self.feed_forward_params_per_layer()
            + self.layernorm_params_per_layer()
        )
        breakdown = {
            "embedding": self.embedding_params(),
            "attention": self.attention_params_per_layer() * self.n_layers,
            "feed_forward": self.feed_forward_params_per_layer() * self.n_layers,
            "layernorm": self.layernorm_params_per_layer() * self.n_layers,
            "output_head": self.output_head_params(),
        }
        breakdown["total"] = sum(v for k, v in breakdown.items() if k != "total")
        return breakdown


print(ModelConfig(vocab_size=1000, d_model=64, n_layers=2, n_heads=4, d_ff=256).count_parameters())

## Real-world project

The cells below work out parameter counts for the real GPT-2 designs. Then they compare them with the official totals. GPT-2's output head is tied to the input embedding. That means they share one matrix. So we take away the extra copy to match the real numbers. Expect the computed total to land within about 1% of the official one, not exactly on it. Our formula leaves out positional embeddings and bias vectors, to keep the exercise simple. Almost all of the leftover difference comes from those two things.

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline

# Real, published GPT-2 configs (Radford et al., 2019) and their real total parameter counts.
REAL_CONFIGS = {
    "GPT-2 small": (ModelConfig(vocab_size=50257, d_model=768, n_layers=12, n_heads=12, d_ff=3072), 124_439_808),
    "GPT-2 medium": (ModelConfig(vocab_size=50257, d_model=1024, n_layers=24, n_heads=16, d_ff=4096), 354_823_168),
}

names, computed, published = [], [], []
for name, (config, real_total) in REAL_CONFIGS.items():
    breakdown = config.count_parameters()
    # GPT-2 ties the output head to the embedding table, so the real model doesn't
    # pay for both - subtract the tied duplicate to compare apples to apples.
    tied_total = breakdown["total"] - breakdown["output_head"]
    names.append(name)
    computed.append(tied_total)
    published.append(real_total)
    print(f"{name}: computed(tied)={tied_total:,}  published={real_total:,}  "
          f"diff={abs(tied_total - real_total):,}")
    print(f"  breakdown: {breakdown}")

x = range(len(names))
plt.figure(figsize=(7, 5))
plt.bar([i - 0.2 for i in x], computed, width=0.4, label="computed (tied)")
plt.bar([i + 0.2 for i in x], published, width=0.4, label="published")
plt.xticks(list(x), names)
plt.ylabel("parameters")
plt.title("Computed vs. published parameter counts")
plt.legend()
plt.tight_layout()
plt.show()